# 9.2 Logit lens 与直接 logit 归因 / The logit lens and direct logit attribution

残差流是各层输出之和。把中间层的残差流直接解码成词表上的分布，可以看到预测怎样逐层形成，把最终的 logit 拆开，可以看到每个组件直接贡献了多少。

The residual stream is the sum of the outputs of all layers. Decoding it at an intermediate layer into a distribution over the vocabulary shows how a prediction forms layer by layer, and splitting the final logit shows how much each component contributes directly.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现 logit lens / Implement the logit lens

用前向钩子取出最后一个位置上嵌入之后和每个块之后的残差流，用最终的 LayerNorm 和输出矩阵解码，并计算某个 token 在各层的名次以及各层分布与最终分布之间的 KL 散度。测试检查取出的状态与模型报告的一致、最后一行解码结果等于模型输出的 logits，以及 Paris 的名次随层数的变化。请查看打印出的逐层结果和 9 条提示的对比。

Capture with forward hooks the residual stream at the last position after the embedding and after every block, decode it with the final LayerNorm and the output matrix, and compute the rank of a token at every layer and the KL divergence between the distribution of every layer and the final one. The tests check that the captured states agree with those the model reports, that decoding the last row gives the logits the model outputs, and how the rank of Paris changes with depth. Look at the printed layer-wise results and the comparison of 9 prompts.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    return model, tok


@torch.no_grad()
def residual_stream(model, ids):
    """The residual stream at the last position after the embedding and after each block.

    ids: LongTensor (T,). Returns a tensor (n_layer + 1, d_model). Row 0 is token embedding plus
    position embedding, row l is the output of block l - 1, before the final LayerNorm.
    """
    states = []
    handles = []
    # TODO 1: register a forward hook on model.transformer.drop (its output is the embedding sum)
    #         that appends out[0, -1] to states. A forward hook has the signature hook(module, args, output).
    # TODO 2: register a forward hook on every block in model.transformer.h. The output of a block
    #         can be a tuple whose first element is the hidden state.
    # TODO 3: run the model once, remove the handles, stack the states.
    #         Do not use output_hidden_states here: its last entry already has the final LayerNorm applied.
    raise NotImplementedError


@torch.no_grad()
def logit_lens(model, resid):
    """Decode every row of resid (n, d_model) with the final LayerNorm and the unembedding. Returns (n, V)."""
    # TODO 4: model.transformer.ln_f, then model.lm_head
    raise NotImplementedError


def token_rank(logits, token_id):
    """Rank of a token in each row of logits (n, V): 1 means it is the top prediction. Returns a list."""
    # TODO 5: one plus the number of tokens with a strictly larger logit
    raise NotImplementedError


def kl_to_final(logits):
    """KL(final || layer) in nats for every row, where final is the distribution of the last row."""
    # TODO 6: sum_v p_final(v) * (log p_final(v) - log p_layer(v))
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

model, tok = load_gpt2()
enc = lambda s: tok(s, return_tensors="pt").input_ids[0]

prompt = "The Eiffel Tower is located in the city of"
ids = enc(prompt)
resid = residual_stream(model, ids)

# 1. Thirteen states of size 768, and no hooks left behind.
assert resid.shape == (13, 768), resid.shape
assert len(model.transformer.drop._forward_hooks) == 0 and all(len(b._forward_hooks) == 0 for b in model.transformer.h)

# 2. Row 0 is the embedding sum, the rows agree with the hidden states the model reports.
with torch.no_grad():
    out = model(ids[None], output_hidden_states=True)
    emb = model.transformer.wte(ids[-1]) + model.transformer.wpe(torch.tensor(len(ids) - 1))
assert torch.allclose(resid[0], emb, atol=1e-5)
for l in range(12):
    assert torch.allclose(resid[l], out.hidden_states[l][0, -1], atol=1e-4), l
assert not torch.allclose(resid[12], out.hidden_states[12][0, -1], atol=1e-2), "row 12 is taken before the final LayerNorm"

# 3. The lens applied to the last state is exactly the output of the model.
lens = logit_lens(model, resid)
assert lens.shape == (13, 50257)
assert torch.allclose(lens[-1], out.logits[0, -1], atol=1e-3), "the last row of the lens equals the logits of the model"

# 4. Ranks and KL.
toy = torch.tensor([[1.0, 3.0, 2.0], [5.0, 0.0, 1.0]])
assert token_rank(toy, 0) == [3, 1] and token_rank(toy, 2) == [2, 2]
kl = kl_to_final(toy)
p, q = torch.softmax(toy[1], -1), torch.softmax(toy[0], -1)
assert abs(kl[0] - (p * (p / q).log()).sum().item()) < 1e-5 and abs(kl[1]) < 1e-6

paris = enc(" Paris")[0].item()
ranks = token_rank(lens, paris)
kls = kl_to_final(lens)
assert ranks[-1] == 1, "GPT-2 small predicts Paris here"
assert ranks[0] > 1000, "the answer is not readable from the embedding"
assert min(l for l in range(13) if ranks[l] <= 10) >= 6, "the answer enters the top 10 only in the upper half"
assert abs(kls[-1]) < 1e-5 and kls[0] > kls[-2]

print(prompt)
print("layer  top token        rank of ' Paris'   p(' Paris')   KL to final")
for l in range(13):
    top = tok.decode(lens[l].argmax())
    pr = torch.softmax(lens[l], -1)[paris].item()
    print(f"{l:>5}  {top!r:<16} {ranks[l]:>8} {pr:>16.4f} {kls[l]:>12.2f}")

# More prompts: the layer at which the final answer first becomes the top prediction.
prompts = ["The Statue of Liberty is located in the city of New", "Apple was founded by Steve", "Monday, Tuesday, Wednesday,",
           "The currency of Japan is the", "The Brandenburg Gate is located in the city of",
           "When Mary and John went to the store, John gave a drink to", "Barack Obama was born in the state of",
           "The language spoken in Spain is", "The Colosseum is located in the city of"]
print()
print("prompt | final answer | first layer at which it is top | top token at layers 9, 10, 11")
firsts = {}
for p in prompts:
    L = logit_lens(model, residual_stream(model, enc(p)))
    ans = L[-1].argmax().item()
    r = token_rank(L, ans)
    firsts[p] = min(l for l in range(13) if r[l] == 1)
    print(f"{p} | {tok.decode(ans)!r} | {firsts[p]} | " + " ".join(repr(tok.decode(L[l].argmax())) for l in (9, 10, 11)))
assert firsts[prompts[0]] <= 2, "a fixed two-word name is completed almost from the embedding"
assert firsts[prompts[4]] >= 8, "a fact about a named entity appears late"

<details><summary><b>提示 1 / Hint 1</b></summary>

前向钩子的签名是 `hook(module, args, output)`。`model.transformer.drop` 的输出是张量，块的输出可能是元组，写成 `out[0] if isinstance(out, tuple) else out`。

A forward hook has the signature `hook(module, args, output)`. The output of `model.transformer.drop` is a tensor and the output of a block can be a tuple, so write `out[0] if isinstance(out, tuple) else out`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

钩子按前向计算的顺序被调用，往同一个列表里 `append` 即可得到按层排列的 13 个状态。运行完成后对每个句柄调用 `remove()`。

Hooks are called in the order of the forward pass, so appending to one list gives the 13 states in layer order. Call `remove()` on every handle after the run.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

名次是 `(logits > logits[:, token_id][:, None]).sum(-1) + 1`。

The rank is `(logits > logits[:, token_id][:, None]).sum(-1) + 1`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_gpt2():
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    return model, tok


@torch.no_grad()
def residual_stream(model, ids):
    """The residual stream at the last position after the embedding and after each block.

    ids: LongTensor (T,). Returns a tensor (n_layer + 1, d_model). Row 0 is token embedding plus
    position embedding, row l is the output of block l - 1, before the final LayerNorm.
    """
    states = []
    handles = [model.transformer.drop.register_forward_hook(lambda m, a, out: states.append(out[0, -1]))]
    for block in model.transformer.h:
        handles.append(block.register_forward_hook(
            lambda m, a, out: states.append((out[0] if isinstance(out, tuple) else out)[0, -1])))
    try:
        model(ids[None])
    finally:
        for h in handles:
            h.remove()
    return torch.stack(states)


@torch.no_grad()
def logit_lens(model, resid):
    """Decode every row of resid (n, d_model) with the final LayerNorm and the unembedding. Returns (n, V)."""
    return model.lm_head(model.transformer.ln_f(resid))


def token_rank(logits, token_id):
    """Rank of a token in each row of logits (n, V): 1 means it is the top prediction. Returns a list."""
    target = logits[:, token_id][:, None]
    return ((logits > target).sum(-1) + 1).tolist()


def kl_to_final(logits):
    """KL(final || layer) in nats for every row, where final is the distribution of the last row."""
    logp = torch.log_softmax(logits, -1)
    final = logp[-1]
    return (final.exp() * (final - logp)).sum(-1).tolist()
```

</details>

## 2. 直接 logit 归因 / Direct logit attribution

用钩子取出构成残差流的 25 项，即嵌入以及每一层注意力子层和前馈子层的输出，再按推导 2 把两个 token 的 logit 差分解为各项的贡献。测试检查 25 项之和等于残差流，各项贡献之和加上常数等于模型实际的 logit 差。

Capture with hooks the 25 terms that make up the residual stream, namely the embedding and the outputs of the attention sublayer and the feed-forward sublayer of every layer, and split the logit difference of two tokens into the contributions of the terms following derivation 2. The tests check that the 25 terms sum to the residual stream and that the contributions plus the constant equal the actual logit difference of the model.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


@torch.no_grad()
def components(model, ids):
    """The terms whose sum is the residual stream at the last position before the final LayerNorm.

    Returns (names, parts) with parts of shape (1 + 2 * n_layer, d_model): the embedding, then for
    each layer the output of its attention sublayer and of its MLP.
    """
    names, parts = ["embed"], []
    handles = []
    # TODO 1: a forward hook on model.transformer.drop that appends out[0, -1]
    # TODO 2: for every block, hooks on block.attn (its output can be a tuple) and on block.mlp,
    #         and the names f"attn{l}", f"mlp{l}" in the same order
    # TODO 3: run the model, remove the handles, stack
    raise NotImplementedError


@torch.no_grad()
def direct_logit_attribution(model, parts, token_a, token_b):
    """Contribution of every part to logit(token_a) - logit(token_b).

    The final LayerNorm computes g * (x - mean(x)) / std(x) + b. With std(x) of the full residual
    stream held fixed, this is linear in x, so the logit difference splits into one term per part
    plus a constant that comes from b. Returns (contributions (n_parts,), constant).
    """
    ln = model.transformer.ln_f
    # TODO 4: total = parts.sum(0). std = sqrt(mean((total - mean(total))^2) + ln.eps)
    # TODO 5: direction = row token_a minus row token_b of model.lm_head.weight
    # TODO 6: centre every part by its own mean, divide by std, multiply by ln.weight, dot with direction
    # TODO 7: the constant is ln.bias dotted with direction
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

model, tok = load_gpt2()
enc = lambda s: tok(s, return_tensors="pt").input_ids[0]
prompt = "The Eiffel Tower is located in the city of"
ids = enc(prompt)
paris, london = enc(" Paris")[0].item(), enc(" London")[0].item()

names, parts = components(model, ids)

# 1. One embedding term and two terms per layer.
assert names[:3] == ["embed", "attn0", "mlp0"] and names[-1] == "mlp11" and len(names) == 25
assert parts.shape == (25, 768)
assert all(len(b.attn._forward_hooks) == 0 and len(b.mlp._forward_hooks) == 0 for b in model.transformer.h)

# 2. The residual stream is the sum of these terms.
resid = residual_stream(model, ids)
assert torch.allclose(parts.sum(0), resid[-1], atol=1e-3), "the parts add up to the final residual stream"
assert torch.allclose(parts[:5].sum(0), resid[2], atol=1e-3), "and the first 1 + 2l parts to the stream after l blocks"

# 3. The contributions and the constant add up to the logit difference of the model.
contrib, const = direct_logit_attribution(model, parts, paris, london)
with torch.no_grad():
    logits = model(ids[None]).logits[0, -1]
true = (logits[paris] - logits[london]).item()
assert contrib.shape == (25,)
assert abs(contrib.sum().item() + const - true) < 1e-2, (contrib.sum().item() + const, true)

# 4. Antisymmetry.
c2, k2 = direct_logit_attribution(model, parts, london, paris)
assert torch.allclose(c2, -contrib, atol=1e-5) and abs(k2 + const) < 1e-6

print(prompt)
print(f"logit(' Paris') - logit(' London') = {true:.2f}, sum of the parts {contrib.sum().item():.2f} plus constant {const:.2f}")
order = contrib.abs().argsort(descending=True)[:8].tolist()
print("largest direct contributions:")
for j in order:
    print(f"  {names[j]:<7} {contrib[j]:+.2f}")
att = sum(contrib[j].item() for j in range(25) if names[j].startswith("attn"))
mlp = sum(contrib[j].item() for j in range(25) if names[j].startswith("mlp"))
print(f"all attention sublayers {att:+.2f}, all MLPs {mlp:+.2f}, embedding {contrib[0]:+.2f}")

# A second prompt.
p2 = "The opposite of hot is"
cold, hot = enc(" cold")[0].item(), enc(" hot")[0].item()
n2, parts2 = components(model, enc(p2))
c3, k3 = direct_logit_attribution(model, parts2, cold, hot)
with torch.no_grad():
    lg = model(enc(p2)[None]).logits[0, -1]
assert abs(c3.sum().item() + k3 - (lg[cold] - lg[hot]).item()) < 1e-2
print(p2)
print(f"logit(' cold') - logit(' hot') = {(lg[cold] - lg[hot]).item():.2f}, largest parts " +
      ", ".join(f"{n2[j]} {c3[j]:+.2f}" for j in c3.abs().argsort(descending=True)[:6].tolist()))

<details><summary><b>提示 1 / Hint 1</b></summary>

`block.attn` 的输出可能是元组，第一个元素是写入残差流的向量。`block.mlp` 的输出是张量。

The output of `block.attn` can be a tuple whose first element is the vector written into the residual stream. The output of `block.mlp` is a tensor.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

$\sigma$ 用完整的和来算，`total = parts.sum(0)`，`std = ((total - total.mean()) ** 2).mean().add(ln.eps).sqrt()`。

Compute $\sigma$ from the full sum, `total = parts.sum(0)`, `std = ((total - total.mean()) ** 2).mean().add(ln.eps).sqrt()`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

每一项按自己的均值去中心化，`parts - parts.mean(-1, keepdim=True)`，再除以 `std`、乘以 `ln.weight`，最后与方向向量做内积。

Centre every term by its own mean, `parts - parts.mean(-1, keepdim=True)`, divide by `std`, multiply by `ln.weight` and take the inner product with the direction.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


@torch.no_grad()
def components(model, ids):
    """The terms whose sum is the residual stream at the last position before the final LayerNorm.

    Returns (names, parts) with parts of shape (1 + 2 * n_layer, d_model): the embedding, then for
    each layer the output of its attention sublayer and of its MLP.
    """
    names, parts = ["embed"], []
    handles = [model.transformer.drop.register_forward_hook(lambda m, a, out: parts.append(out[0, -1]))]
    for l, block in enumerate(model.transformer.h):
        names += [f"attn{l}", f"mlp{l}"]
        handles.append(block.attn.register_forward_hook(
            lambda m, a, out: parts.append((out[0] if isinstance(out, tuple) else out)[0, -1])))
        handles.append(block.mlp.register_forward_hook(lambda m, a, out: parts.append(out[0, -1])))
    try:
        model(ids[None])
    finally:
        for h in handles:
            h.remove()
    return names, torch.stack(parts)


@torch.no_grad()
def direct_logit_attribution(model, parts, token_a, token_b):
    """Contribution of every part to logit(token_a) - logit(token_b).

    The final LayerNorm computes g * (x - mean(x)) / std(x) + b. With std(x) of the full residual
    stream held fixed, this is linear in x, so the logit difference splits into one term per part
    plus a constant that comes from b. Returns (contributions (n_parts,), constant).
    """
    ln = model.transformer.ln_f
    total = parts.sum(0)
    centered_total = total - total.mean()
    std = torch.sqrt(centered_total.pow(2).mean() + ln.eps)
    direction = model.lm_head.weight[token_a] - model.lm_head.weight[token_b]
    centered = parts - parts.mean(-1, keepdim=True)
    contrib = (centered / std * ln.weight) @ direction
    const = (ln.bias @ direction).item()
    return contrib, const
```

</details>

## 3. 残差流分解与 logit 差 / The decomposition of the residual stream and logit differences

在纸上完成。
(a) 证明对所有 logit 加上同一个常数 $c$ 不改变 softmax 的输出，并据此说明两个 token 的概率之比只取决于它们的 logit 差。
(b) 设 $d = 3$，$g = (1, 1, 1)$，$b = 0$，忽略 $\epsilon$。残差流由两项组成，$x_1 = (2, 0, -2)$，$x_2 = (1, 1, -2)$。方向 $u = w_a - w_b = (1, 0, -1)$。计算 $\sigma$、两项各自的贡献以及 logit 差，并验证贡献之和等于直接计算的结果。
(c) 说明为什么 $\operatorname{LN}(\operatorname{LN}(x))$ 一般不等于 $\operatorname{LN}(x)$，并指出在什么特殊情况下两者相等。
(d) 说明 logit lens 在第 $k$ 层之后的结果，等于把完整模型中第 $k$ 层及以后所有子层的输出置零后得到的输出。再说明它为什么不等于“模型在只有 $k$ 层时会输出什么”。

Work on paper.
(a) Prove that adding the same constant $c$ to all logits leaves the output of the softmax unchanged, and conclude that the ratio of the probabilities of two tokens depends only on the difference of their logits.
(b) Let $d = 3$, $g = (1, 1, 1)$, $b = 0$ and ignore $\epsilon$. The residual stream consists of two terms, $x_1 = (2, 0, -2)$ and $x_2 = (1, 1, -2)$. The direction is $u = w_a - w_b = (1, 0, -1)$. Compute $\sigma$, the contribution of each term and the logit difference, and verify that the contributions sum to the directly computed result.
(c) Explain why $\operatorname{LN}(\operatorname{LN}(x))$ in general differs from $\operatorname{LN}(x)$, and name the special case in which the two agree.
(d) Show that the result of the logit lens after layer $k$ equals the output obtained by zeroing in the full model the outputs of all sublayers from layer $k$ on. Then explain why it is not what a model with only $k$ layers would output.

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) 先求和 $x = x_1 + x_2$，再算它的均值和标准差。

(b) Sum $x = x_1 + x_2$ and then compute its mean and standard deviation.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 考虑 $\operatorname{LN}(x)$ 的均值和标准差分别是多少。

(c) Consider what the mean and the standard deviation of $\operatorname{LN}(x)$ are.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 使用推导 1 的最后一步。

(d) Use the last step of derivation 1.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $\frac{e^{z_v + c}}{\sum_k e^{z_k + c}} = \frac{e^{c} e^{z_v}}{e^{c} \sum_k e^{z_k}} = \frac{e^{z_v}}{\sum_k e^{z_k}}$。两个 token 的概率之比为 $\frac{p_a}{p_b} = \frac{e^{z_a}}{e^{z_b}} = e^{z_a - z_b}$，分母相同而消去，所以只取决于 logit 差。

(b) $x = (3, 1, -4)$，$\mu(x) = 0$，$\sigma = \sqrt{(9 + 1 + 16)/3} = \sqrt{26/3} \approx 2.944$。两项的均值分别为 $\mu(x_1) = 0$ 和 $\mu(x_2) = 0$，无需再去中心化。$u^\top x_1 = 2 + 2 = 4$，贡献为 $4 / 2.944 \approx 1.359$。$u^\top x_2 = 1 + 2 = 3$，贡献为 $3 / 2.944 \approx 1.019$。两者之和约为 $2.378$。直接计算，$\operatorname{LN}(x) = (3, 1, -4) / 2.944$，$u^\top \operatorname{LN}(x) = (3 + 4) / 2.944 \approx 2.378$，两者一致。

(c) $\operatorname{LN}(x)$ 的均值是 $\mu(g \odot \hat{x} + b)$，标准差是 $\sigma(g \odot \hat{x} + b)$，其中 $\hat{x}$ 是标准化后的向量。由于 $g$ 的各分量不同，$b$ 也不为零，这个向量的均值一般不是 0，标准差一般不是 1，再做一次归一化会重新平移和缩放，并再乘一次 $g$、再加一次 $b$。只有当 $g$ 的各分量都等于 1 且 $b = 0$ 时，$\operatorname{LN}(x) = \hat{x}$ 的均值为 0、标准差为 1（忽略 $\epsilon$），第二次归一化才不改变它。训练好的模型不满足这个条件。

(d) 由推导 1，$x^{(L)} = x^{(0)} + \sum_{\ell < k} (A^{(\ell)} + F^{(\ell)}) + \sum_{\ell \ge k} (A^{(\ell)} + F^{(\ell)})$。把第二个和中的各项置零后，最后的残差流就是 $x^{(k)}$，再经过最终的 LayerNorm 和输出矩阵，得到的正是 logit lens 的结果。一个只有 $k$ 层的模型则会在训练中学到让第 $k$ 层的输出直接适合被解码，它的权重与 12 层模型的前 $k$ 层不同。12 层模型的第 $k$ 层是为后面的层准备输入的，不是为输出矩阵准备的。

---

(a) $\frac{e^{z_v + c}}{\sum_k e^{z_k + c}} = \frac{e^{c} e^{z_v}}{e^{c} \sum_k e^{z_k}} = \frac{e^{z_v}}{\sum_k e^{z_k}}$. The ratio of the probabilities of two tokens is $\frac{p_a}{p_b} = \frac{e^{z_a}}{e^{z_b}} = e^{z_a - z_b}$, since the common denominator cancels, so it depends on the logit difference alone.

(b) $x = (3, 1, -4)$, $\mu(x) = 0$, $\sigma = \sqrt{(9 + 1 + 16)/3} = \sqrt{26/3} \approx 2.944$. The means of the two terms are $\mu(x_1) = 0$ and $\mu(x_2) = 0$, so no further centring is needed. $u^\top x_1 = 2 + 2 = 4$ gives a contribution of $4 / 2.944 \approx 1.359$. $u^\top x_2 = 1 + 2 = 3$ gives $3 / 2.944 \approx 1.019$. Their sum is about $2.378$. Directly, $\operatorname{LN}(x) = (3, 1, -4) / 2.944$ and $u^\top \operatorname{LN}(x) = (3 + 4) / 2.944 \approx 2.378$, in agreement.

(c) The mean of $\operatorname{LN}(x)$ is $\mu(g \odot \hat{x} + b)$ and its standard deviation is $\sigma(g \odot \hat{x} + b)$, where $\hat{x}$ is the standardized vector. Since the components of $g$ differ and $b$ is not zero, this vector in general has neither mean 0 nor standard deviation 1, so a second normalization shifts and scales it again and applies $g$ and $b$ once more. Only when all components of $g$ equal 1 and $b = 0$ does $\operatorname{LN}(x) = \hat{x}$ have mean 0 and standard deviation 1 (ignoring $\epsilon$), and the second normalization leaves it unchanged. A trained model does not satisfy this condition.

(d) By derivation 1, $x^{(L)} = x^{(0)} + \sum_{\ell < k} (A^{(\ell)} + F^{(\ell)}) + \sum_{\ell \ge k} (A^{(\ell)} + F^{(\ell)})$. With the terms of the second sum set to zero the last residual stream is $x^{(k)}$, and passing it through the final LayerNorm and the output matrix gives exactly the result of the logit lens. A model with only $k$ layers would instead learn in training to make the output of layer $k$ directly suitable for decoding, and its weights differ from the first $k$ layers of the 12-layer model. Layer $k$ of the 12-layer model prepares input for the later layers and not for the output matrix.

</details>